In [ ]:
import os
import pickle
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

raw_root = Path(os.environ["GAITLU_RAW_ROOT"]).expanduser()
output_root = (
    Path(os.environ["GAITLU_OUTPUT_ROOT"]).expanduser()
    / "visualizations"
)
shard_name = os.environ.get("GAITLU_SHARD", "gaitlu-000")

# Raw GaitLU shards are normally named gaitlu-000.tar.gz.
candidates = [
    raw_root / f"{shard_name}.tar.gz",
    raw_root / f"{shard_name}.tgz",
    raw_root / f"{shard_name}.tar",
]
tar_path = next((path for path in candidates if path.is_file()), None)

if tar_path is None:
    matches = sorted(raw_root.glob(f"{shard_name}*.tar*"))
    if matches:
        tar_path = matches[0]
    else:
        raise FileNotFoundError(
            f"No .tar/.tar.gz shard found for {shard_name!r} under {raw_root}"
        )

print(f"Reading raw shard: {tar_path}")


def decode_sequence(value):
    if isinstance(value, dict):
        for key in ("silhouettes", "frames", "data"):
            if key in value:
                value = value[key]
                break
        else:
            raise ValueError(
                f"Pickle dictionary has no recognized frame key: {list(value)}"
            )

    if isinstance(value, (list, tuple)) and len(value) == 1:
        candidate = np.asarray(value[0])
        if candidate.ndim >= 3:
            value = candidate

    frames = np.asarray(value)

    if frames.ndim == 4 and frames.shape[1] == 1:
        frames = frames[:, 0]
    elif frames.ndim == 4 and frames.shape[-1] == 1:
        frames = frames[..., 0]

    if frames.ndim != 3 or 0 in frames.shape:
        raise ValueError(f"Expected non-empty [T, H, W], received {frames.shape}")

    if frames.max() <= 1:
        return frames >= 0.5
    return frames >= 128


videos = []
# r|* streams .tar, .tar.gz, and .tgz without indexing the whole archive.
with tarfile.open(tar_path, mode="r|*") as archive:
    for member in archive:
        if not member.isfile() or not member.name.lower().endswith(".pkl"):
            continue

        handle = archive.extractfile(member)
        if handle is None:
            raise OSError(f"Could not read tar member: {member.name}")

        with handle:
            # Only unpickle files from the trusted GaitLU release.
            value = pickle.load(handle)
        videos.append((member.name, decode_sequence(value)))

        if len(videos) == 4:
            break

if not videos:
    raise RuntimeError(
        "The shard contains no .pkl members. It may be a prepared bit-packed "
        "shard containing records/*.bits, which requires metadata."
    )

print("Loaded sequences:")
for member_name, video in videos:
    print(f"  {member_name}: shape={video.shape}, dtype={video.dtype}")

frames_to_show = 8
fig, axes = plt.subplots(
    len(videos),
    frames_to_show,
    figsize=(16, 2.8 * len(videos)),
    squeeze=False,
)

for row_index, (member_name, video) in enumerate(videos):
    frame_indices = np.linspace(0, len(video) - 1, frames_to_show).astype(int)

    for column_index, frame_index in enumerate(frame_indices):
        ax = axes[row_index, column_index]
        ax.imshow(
            video[frame_index],
            cmap="gray",
            vmin=0,
            vmax=1,
            interpolation="nearest",
        )
        ax.set_title(f"frame {frame_index}")
        ax.axis("off")

    axes[row_index, 0].set_ylabel(
        Path(member_name).parent.name,
        rotation=0,
        labelpad=35,
        va="center",
    )

fig.suptitle(f"{shard_name}: sample GaitLU silhouettes")
fig.tight_layout()

output_root.mkdir(parents=True, exist_ok=True)
png_path = output_root / f"{shard_name}.png"
fig.savefig(png_path, dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"Saved PNG: {png_path}")
display(Image(filename=str(png_path)))
